<a href="https://colab.research.google.com/github/Benxperia/MRes/blob/main/TVB_LScomp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Set up


In [ ]:
#mount Google Drive
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

Mounted at /content/drive/


In [ ]:
import ee
import time
import pandas as pd
import numpy as np
import geopandas as gpd

In [ ]:
# Trigger the authentication flow.
ee.Authenticate()

# Initialize the API.
ee.Initialize(project='rs-s2-450716')

#Load shapefile from drive

In [ ]:
# Load the shapefile
shapefile_path = "/content/drive/MyDrive/MRes/Landcover analysis/shp/TRVB_upstream_lvl6_dissolve_repro.shp"
gdf = gpd.read_file(shapefile_path)

# Convert to WGS84 if not already
if gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    print(f"Reprojected from {gdf.crs} to EPSG:4326")

# Get the geometry and convert to Earth Engine format
geom = gdf.geometry.iloc[0]  # Get first polygon
coords = list(geom.exterior.coords)

# Create Earth Engine geometry
AOI = ee.Geometry.Polygon(coords)

# Get the bounds for reference
bounds = gdf.total_bounds  # [minx, miny, maxx, maxy]
print(f"AOI bounds: {bounds}")
print(f"Center approximately: {(bounds[1]+bounds[3])/2}, {(bounds[0]+bounds[2])/2}")

Reprojected from EPSG:4326 to EPSG:4326
AOI bounds: [-147.68755569   61.70831914 -140.36662622   65.12147741]
Center approximately: 63.41489827306803, -144.0270909539061


#Location and parameters

In [ ]:
# Site name for the image
ImageName = "Alaska_TRV_Basin"

# Define years of interest
startYear = 1978
endYear = 2024

# Cloud tolerance in %
Cloud_Max = 30

print(f"Processing area: {ImageName}")
print(f"Years: {startYear} to {endYear}")

'''Processing Functions'''
# Function to check if a collection has images for a given year and area
def hasImagesForYear(collection_id, year, date_range='standard'):
    # Define date ranges based on parameter
    if date_range == 'standard':
        start_month, end_month = 8, 9  # August-September
    elif date_range == 'expanded':
        start_month, end_month = 6, 10  # June-October
    elif date_range == 'full_summer':
        start_month, end_month = 5, 10  # May-October

    startDate = ee.Date.fromYMD(year, start_month, 1)
    endDate = ee.Date.fromYMD(year, end_month, 30)

    collection = ee.ImageCollection(collection_id)
    filtered = collection.filterDate(startDate, endDate) \
                        .filterBounds(AOI) \
                        .filter(ee.Filter.lt('CLOUD_COVER', Cloud_Max))

    count = filtered.size().getInfo()
    return count > 0

# Custom cloud mask function for glacier studies
def customCloudMask(image):
    qa = image.select('QA_PIXEL')
    # Bit 3: Cloud, Bit 4: Cloud shadow
    cloud = qa.bitwiseAnd(1 << 3).neq(0)
    shadow = qa.bitwiseAnd(1 << 4).neq(0)

    # Mask where cloud or shadow is present
    mask = cloud.Or(shadow).Not()
    return image.updateMask(mask)

# Function to process Landsat MSS (1-3) data - expanded to 7 bands
def processMSS(year, cloud_max, startDate, endDate):
    # Use the MSS collection
    collection = ee.ImageCollection('LANDSAT/LM03/C02/T2')

    # Filter by date and cloud cover
    filtered = collection.filterDate(startDate, endDate) \
                        .filterBounds(AOI) \
                        .filter(ee.Filter.lt('CLOUD_COVER', cloud_max))

    # If no images, return null
    count = filtered.size().getInfo()
    if count == 0:
        print(f"No MSS images found for {year}")
        return None

    print(f"Found {count} MSS images for {year}")

    # Create composite - MSS has only 4 bands, so we'll duplicate some to get to 7
    composite = filtered.median()

    # Get original bands
    b4 = composite.select('B4')  # Green
    b5 = composite.select('B5')  # Red
    b6 = composite.select('B6')  # Red
    b7 = composite.select('B7')  # NIR

    # Create a 7-band image by duplicating existing bands
    selected = ee.Image.cat([
        b7, b6, b4, b7, b6, b4, b7
    ]).rename(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7'])

    # Fill any gaps using max values
    max_composite = filtered.max()
    b4_max = max_composite.select('B4')
    b5_max = max_composite.select('B5')
    b6_max = max_composite.select('B6')
    b7_max = max_composite.select('B7')

    max_filled = ee.Image.cat([
        b7_max, b6_max, b4_max, b7_max, b6_max, b4_max, b7_max
    ]).rename(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7'])

    selected = selected.unmask(max_filled)

    return selected.set('year', year)

# Function to process Landsat TM, ETM+ (4-7)
def processLandsat4to7(year, collection_id, cloud_max, startDate, endDate):
    collection = ee.ImageCollection(collection_id)

    filtered = collection.filterDate(startDate, endDate) \
                        .filterBounds(AOI) \
                        .filter(ee.Filter.lt('CLOUD_COVER', cloud_max))

    # If no images, return null
    count = filtered.size().getInfo()
    if count == 0:
        print(f"No images found in {collection_id} for {year}")
        return None

    print(f"Found {count} images in {collection_id} for {year}")

    # Apply custom cloud masking
    masked = filtered.map(customCloudMask)

    # Create median composite
    composite = masked.median()

    # Select bands and duplicate one to get 7 bands
    selected = composite.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'])
    b4_duplicate = composite.select('SR_B4')
    selected = ee.Image.cat([selected, b4_duplicate]).rename(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7'])

    # Fill gaps with max values
    max_composite = masked.max()
    max_selected = max_composite.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'])
    max_b4_duplicate = max_composite.select('SR_B4')
    max_filled = ee.Image.cat([max_selected, max_b4_duplicate]).rename(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7'])

    selected = selected.unmask(max_filled)

    return selected.set('year', year)

# Function to process Landsat 7 SLC-off data
def processLandsat7SLCOff(year, collection_id, cloud_max, startDate, endDate):
    collection = ee.ImageCollection(collection_id)

    filtered = collection.filterDate(startDate, endDate) \
                        .filterBounds(AOI) \
                        .filter(ee.Filter.lt('CLOUD_COVER', cloud_max))

    count = filtered.size().getInfo()
    if count == 0:
        print(f"No images found in {collection_id} for {year}")
        return None

    print(f"Found {count} Landsat 7 SLC-off images for {year}")

    # Apply custom cloud masking
    masked = filtered.map(customCloudMask)

    # For SLC-off data, use specialized gap filling
    median_composite = masked.median()

    # Use max composite to fill gaps
    max_composite = masked.max()
    median_composite = median_composite.unmask(max_composite)

    # Use focal mean to fill small remaining gaps
    kernel = ee.Kernel.circle(radius=3)
    focal_mean = median_composite.focal_mean(kernel=kernel, iterations=3)
    median_composite = median_composite.unmask(focal_mean)

    # Select and rename bands
    selected = median_composite.select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7'])
    b4_duplicate = median_composite.select('SR_B4')
    selected = ee.Image.cat([selected, b4_duplicate]).rename(['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7'])

    return selected.set('year', year)

# Function to process Landsat OLI (8-9)
def processLandsat8to9(year, collection_id, cloud_max, startDate, endDate):
    collection = ee.ImageCollection(collection_id)

    filtered = collection.filterDate(startDate, endDate) \
                        .filterBounds(AOI) \
                        .filter(ee.Filter.lt('CLOUD_COVER', cloud_max))

    # If no images, return null
    count = filtered.size().getInfo()
    if count == 0:
        print(f"No images found in {collection_id} for {year}")
        return None

    print(f"Found {count} images in {collection_id} for {year}")

    # Apply custom cloud masking
    masked = filtered.map(customCloudMask)

    # Create median composite
    composite = masked.median()

    # Rename to consistent band naming
    selected = composite.select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7']
    )

    # Fill gaps with max values
    max_composite = masked.max().select(
        ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'],
        ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7']
    )
    selected = selected.unmask(max_composite)

    return selected.set('year', year)

# Export function with enhanced gap filling
def exportImageWithFill(image, filename):
    if image is None:
        print(f"No data available for {filename}")
        return False

    try:
        # Apply progressive filling approach
        kernel_small = ee.Kernel.circle(radius=1)
        filled_mean_small = image.focal_mean(kernel=kernel_small, iterations=1)
        image_filled = image.unmask(filled_mean_small)

        kernel_medium = ee.Kernel.circle(radius=2)
        filled_mean_medium = image_filled.focal_mean(kernel=kernel_medium, iterations=2)
        image_filled = image_filled.unmask(filled_mean_medium)

        kernel_large = ee.Kernel.circle(radius=3)
        filled_mean_large = image_filled.focal_mean(kernel=kernel_large, iterations=3)
        image_filled = image_filled.unmask(filled_mean_large)

        # Convert to Float32
        image_filled = image_filled.toFloat()

        folder = 'Alaska_TRV_Landsat'
        fileNamePrefix = filename

        export_task = ee.batch.Export.image.toDrive(
            image=image_filled,
            description=filename,
            folder=folder,
            fileNamePrefix=fileNamePrefix,
            region=AOI,
            scale=30,
            maxPixels=1e13,
            fileFormat='GeoTIFF'
        )

        export_task.start()
        print(f"Started export for {filename}")
        return True

    except Exception as e:
        print(f"Error exporting {filename}: {e}")
        return False

# Function to find the best available collection for a year
def findBestCollection(year, cloud_max=30, date_range='standard'):
    # Define date ranges
    if date_range == 'standard':
        start_month, end_month = 8, 9
    elif date_range == 'expanded':
        start_month, end_month = 6, 10
    elif date_range == 'full_summer':
        start_month, end_month = 5, 10

    # Check collections in prioritized order
    if year >= 2021:
        if hasImagesForYear('LANDSAT/LC09/C02/T1_L2', year, date_range):
            return 'LANDSAT/LC09/C02/T1_L2'
        if hasImagesForYear('LANDSAT/LC08/C02/T1_L2', year, date_range):
            return 'LANDSAT/LC08/C02/T1_L2'
    elif year >= 2013:
        if hasImagesForYear('LANDSAT/LC08/C02/T1_L2', year, date_range):
            return 'LANDSAT/LC08/C02/T1_L2'
    elif year >= 1999 and year <= 2012:
        if hasImagesForYear('LANDSAT/LT05/C02/T1_L2', year, date_range):
            return 'LANDSAT/LT05/C02/T1_L2'
        elif hasImagesForYear('LANDSAT/LE07/C02/T1_L2', year, date_range):
            return 'LANDSAT/LE07/C02/T1_L2'
    elif year >= 1984 and year <= 1998:
        if hasImagesForYear('LANDSAT/LT05/C02/T1_L2', year, date_range):
            return 'LANDSAT/LT05/C02/T1_L2'
        if hasImagesForYear('LANDSAT/LT04/C02/T1_L2', year, date_range):
            return 'LANDSAT/LT04/C02/T1_L2'
    elif year >= 1972 and year <= 1983:
        if hasImagesForYear('LANDSAT/LM03/C02/T2', year, date_range):
            return 'LANDSAT/LM03/C02/T2'

    return None

Processing area: Alaska_TRV_Basin
Years: 1978 to 2024


#Process and export

In [ ]:
# Process each year
for year in range(startYear, endYear + 1):
    print(f"\n{'='*50}")
    print(f"Processing year {year}...")
    filename = f"{ImageName}_LS_Comp_{year}"

    # Find best collection
    collection_id = findBestCollection(year)

    if collection_id is None:
        print(f"No suitable Landsat data found for {year}")
        continue

    print(f"Using collection: {collection_id}")

    # Define date ranges
    if collection_id == 'LANDSAT/LM03/C02/T2':
        startDate = ee.Date.fromYMD(year, 8, 1)
        endDate = ee.Date.fromYMD(year, 9, 30)
    else:
        startDate = ee.Date.fromYMD(year, 7, 1)
        endDate = ee.Date.fromYMD(year, 9, 30)

    # Process based on collection type
    if collection_id == 'LANDSAT/LE07/C02/T1_L2' and year >= 2003:
        print("Using SLC-off specific processing for Landsat 7")
        image = processLandsat7SLCOff(year, collection_id, Cloud_Max, startDate, endDate)
    elif collection_id == 'LANDSAT/LM03/C02/T2':
        image = processMSS(year, Cloud_Max, startDate, endDate)
    elif collection_id.startswith('LANDSAT/LC'):
        image = processLandsat8to9(year, collection_id, Cloud_Max, startDate, endDate)
    else:
        image = processLandsat4to7(year, collection_id, Cloud_Max, startDate, endDate)

    # Export
    if image is not None:
        exportImageWithFill(image, filename)

    # Delay between exports
    time.sleep(5)

print("\n" + "="*50)
print("Processing complete!")


Processing year 1978...
Using collection: LANDSAT/LM03/C02/T2
Found 30 MSS images for 1978
Started export for Alaska_TRV_Basin_LS_Comp_1978

Processing year 1979...
Using collection: LANDSAT/LM03/C02/T2
Found 28 MSS images for 1979
Started export for Alaska_TRV_Basin_LS_Comp_1979

Processing year 1980...
Using collection: LANDSAT/LM03/C02/T2
Found 10 MSS images for 1980
Started export for Alaska_TRV_Basin_LS_Comp_1980

Processing year 1981...
Using collection: LANDSAT/LM03/C02/T2
Found 2 MSS images for 1981
Started export for Alaska_TRV_Basin_LS_Comp_1981

Processing year 1982...
Using collection: LANDSAT/LM03/C02/T2
Found 11 MSS images for 1982
Started export for Alaska_TRV_Basin_LS_Comp_1982

Processing year 1983...
No suitable Landsat data found for 1983

Processing year 1984...
Using collection: LANDSAT/LT05/C02/T1_L2
Found 6 images in LANDSAT/LT05/C02/T1_L2 for 1984
Started export for Alaska_TRV_Basin_LS_Comp_1984

Processing year 1985...
Using collection: LANDSAT/LT05/C02/T1_L2


#Task manager

In [ ]:
def checkRecentTasks(hours_ago=72):
    import datetime

    tasks = ee.batch.Task.list()
    current_time = datetime.datetime.now()

    running = 0
    completed = 0
    failed = 0
    ready = 0

    filtered_tasks = []

    print(f"\nTask Status (last {hours_ago} hours):")
    for task in tasks:
        status = task.status()

        if 'creation_timestamp_ms' in status:
            creation_time = datetime.datetime.fromtimestamp(int(status['creation_timestamp_ms'])/1000)
            time_diff = current_time - creation_time

            if time_diff.total_seconds() <= hours_ago * 3600:
                filtered_tasks.append((status, creation_time))

                state = status['state']
                if state == 'RUNNING':
                    running += 1
                elif state == 'COMPLETED':
                    completed += 1
                elif state in ['FAILED', 'CANCELLED']:
                    failed += 1
                elif state == 'READY':
                    ready += 1

    filtered_tasks.sort(key=lambda x: x[1], reverse=True)

    for status, creation_time in filtered_tasks:
        state = status['state']
        creation_str = creation_time.strftime("%Y-%m-%d %H:%M:%S")
        print(f"{creation_str} - {status['description']} - {state}")

        if state in ['FAILED', 'CANCELLED'] and 'error_message' in status:
            print(f"  Error: {status['error_message']}")

    print(f"\nSummary (last {hours_ago} hours): {running} running, {ready} ready, {completed} completed, {failed} failed")

# Check tasks
checkRecentTasks(72)


Task Status (last 72 hours):
2026-01-22 11:00:02 - Alaska_TRV_Basin_LS_Comp_2024 - COMPLETED
2026-01-22 10:59:55 - Alaska_TRV_Basin_LS_Comp_2023 - COMPLETED
2026-01-22 10:59:48 - Alaska_TRV_Basin_LS_Comp_2022 - COMPLETED
2026-01-22 10:59:41 - Alaska_TRV_Basin_LS_Comp_2021 - COMPLETED
2026-01-22 10:59:34 - Alaska_TRV_Basin_LS_Comp_2020 - COMPLETED
2026-01-22 10:59:28 - Alaska_TRV_Basin_LS_Comp_2019 - COMPLETED
2026-01-22 10:59:22 - Alaska_TRV_Basin_LS_Comp_2018 - COMPLETED
2026-01-22 10:59:15 - Alaska_TRV_Basin_LS_Comp_2017 - COMPLETED
2026-01-22 10:59:08 - Alaska_TRV_Basin_LS_Comp_2016 - COMPLETED
2026-01-22 10:59:01 - Alaska_TRV_Basin_LS_Comp_2015 - COMPLETED
2026-01-22 10:58:55 - Alaska_TRV_Basin_LS_Comp_2014 - COMPLETED
2026-01-22 10:58:49 - Alaska_TRV_Basin_LS_Comp_2013 - COMPLETED
2026-01-22 10:58:42 - Alaska_TRV_Basin_LS_Comp_2012 - COMPLETED
2026-01-22 10:58:35 - Alaska_TRV_Basin_LS_Comp_2011 - COMPLETED
2026-01-22 10:58:29 - Alaska_TRV_Basin_LS_Comp_2010 - COMPLETED
2026-01-22